In [0]:
import dlt
from pyspark.sql.functions import col, current_timestamp

In [0]:
bronze_schema_name = spark.conf.get("pipeline.bronze_schema_name", "weather_bronze")
silver_schema_name = spark.conf.get("pipeline.silver_schema_name", "weather_silver")

In [0]:
@dlt.view(name="ly_rainfall_data_cleaned")
#data quality and expectations
@dlt.expect_or_drop("valid_timestamp", "event_time IS NOT NULL")
def ly_rainfall_data_cleaned():
    return (
        dlt.read_stream(f"{bronze_schema_name}.ly_rainfall_data")
        .select(
            col("city"),
            col("lat"),
            col("lon"),
            col("event_timestamp").cast("timestamp").alias("event_time"),
            col("rain_sum"),
            col("showers_sum"),
            col("snowfall_sum"),
            col("source_filename"),
            col("ingestion_timestamp").alias("bronze_ingestion_time"),
            current_timestamp().alias("silver_transformation_time")
        )
    )

dlt.create_streaming_table(
    f"{silver_schema_name}.unified_rainfall",
    table_properties={"quality": "silver"}
)

dlt.apply_changes(
    target=f"{silver_schema_name}.unified_rainfall",
    source="ly_rainfall_data_cleaned",
    keys=["city", "event_time"],         #covers deduplication
    sequence_by="bronze_ingestion_time"  #covers SCD (type 1), upserts the new record
)